# 03 — Backtest Results

This notebook:
1. Runs the full walk-forward backtest over a sample of games.
2. Analyses the performance ledger.
3. Compares full, half, and quarter-Kelly equity curves.
4. Runs a Monte Carlo simulation of future performance ranges.

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

with open('../config/default.yaml') as f:
    cfg = yaml.safe_load(f)

print('Ready.')

## 3.1 Load Data & Fit Models

In [ ]:
from src.data_ingestion.play_by_play import load_pbp
from src.data_ingestion.odds_simulator import SyntheticOddsGenerator
from src.feature_engineering.game_state_features import build_game_state_features
from src.feature_engineering.odds_path_features import build_odds_path_features
from src.models.poisson_scoring import ScoringIntensityModel
from src.models.hawkes_process import HawkesOddsModel

# Load one season of PBP
pbp = load_pbp(seasons=[2022], cache_dir='../data/processed')
pbp_feat = build_game_state_features(pbp)

# Generate synthetic odds for all games
gen = SyntheticOddsGenerator(seed=42)
odds_all = gen.generate_multiple_games(pbp)
odds_feat_all = build_odds_path_features(odds_all)

# Fit scoring model
scoring_model = ScoringIntensityModel(
    avg_seconds_per_play=cfg['poisson_scoring']['avg_seconds_per_play']
).fit(pbp_feat)
print('Scoring model fitted.')

# Fit Hawkes model on the first game (use as representative)
first_gid = pbp_feat['game_id'].unique()[0]
first_game_odds = odds_feat_all[odds_feat_all['game_id'] == first_gid]
event_mask = first_game_odds['odds_return'].abs() > cfg['hawkes']['significant_move_threshold']
event_times = first_game_odds.loc[event_mask, 'elapsed_seconds'].to_numpy()
T = float(first_game_odds['elapsed_seconds'].max())

hawkes_model = HawkesOddsModel(
    threshold=cfg['hawkes']['significant_move_threshold'],
    mu_init=cfg['hawkes']['mu_init'],
    alpha_init=cfg['hawkes']['alpha_init'],
    beta_init=cfg['hawkes']['beta_init'],
).fit(event_times, T)
print(f'Hawkes fitted: μ={hawkes_model.mu_:.4f}, α={hawkes_model.alpha_:.4f}, β={hawkes_model.beta_:.4f}')

## 3.2 Run Walk-Forward Backtest

In [ ]:
from src.backtesting.backtest_engine import WalkForwardBacktest

# Use first 20 games for speed
all_games = list(pbp_feat['game_id'].unique())
test_games = all_games[:20]
print(f'Backtesting {len(test_games)} games...')

kf_params = {
    'Q': cfg['kalman']['Q'],
    'R': cfg['kalman']['R'],
    'x0': cfg['kalman']['x0'],
    'P0': cfg['kalman']['P0'],
    'scoring_Q_multiplier': cfg['kalman']['scoring_Q_multiplier'],
}

engine = WalkForwardBacktest(initial_bankroll=cfg['backtest']['initial_bankroll'])
ledger = engine.run(
    games=test_games,
    pbp=pbp_feat,
    odds=odds_feat_all,
    scoring_model=scoring_model,
    hawkes_model=hawkes_model,
    kf_params=kf_params,
    mc_sims=500,    # reduced for notebook speed; use 5000+ for production
    kelly_fraction=cfg['backtest']['kelly_fraction'],
    edge_threshold=cfg['signals']['edge_threshold'],
)

print(f'\nBacktest complete: {len(ledger)} bets placed.')
print(ledger.head())

In [ ]:
stats = engine.summary_stats(ledger)
print('\n=== Backtest Summary ===')
for k, v in stats.items():
    if isinstance(v, float):
        print(f'  {k:25s}: {v:,.4f}')
    else:
        print(f'  {k:25s}: {v}')

## 3.3 Equity Curve & PnL Distribution

In [ ]:
from src.visualization.plots import plot_backtest_results

if len(ledger) > 0:
    fig = plot_backtest_results(ledger, initial_bankroll=cfg['backtest']['initial_bankroll'])
    plt.show()
else:
    print('No bets placed — try lowering edge_threshold in config/default.yaml')

## 3.4 Kelly Fraction Sensitivity

In [ ]:
from src.visualization.plots import plot_kelly_comparison

initial_bankroll = cfg['backtest']['initial_bankroll']

ledger_full = WalkForwardBacktest(initial_bankroll).run(
    games=test_games, pbp=pbp_feat, odds=odds_feat_all,
    scoring_model=scoring_model, hawkes_model=hawkes_model,
    kf_params=kf_params, mc_sims=500, kelly_fraction=1.0,
)

ledger_half = WalkForwardBacktest(initial_bankroll).run(
    games=test_games, pbp=pbp_feat, odds=odds_feat_all,
    scoring_model=scoring_model, hawkes_model=hawkes_model,
    kf_params=kf_params, mc_sims=500, kelly_fraction=0.5,
)

ledger_quarter = WalkForwardBacktest(initial_bankroll).run(
    games=test_games, pbp=pbp_feat, odds=odds_feat_all,
    scoring_model=scoring_model, hawkes_model=hawkes_model,
    kf_params=kf_params, mc_sims=500, kelly_fraction=0.25,
)

print(f'Full Kelly bets: {len(ledger_full)}')
print(f'Half Kelly bets: {len(ledger_half)}')
print(f'Quarter Kelly bets: {len(ledger_quarter)}')

fig = plot_kelly_comparison(ledger_full, ledger_half, ledger_quarter,
                            initial_bankroll=initial_bankroll)
plt.show()

## 3.5 Edge Distribution Analysis

In [ ]:
if len(ledger) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    ax = axes[0]
    ledger['edge'].hist(bins=30, ax=ax, color='#d62728', edgecolor='white')
    ax.axvline(ledger['edge'].mean(), color='black', ls='--', lw=2,
               label=f"Mean: {ledger['edge'].mean():.3f}")
    ax.set_xlabel('Edge')
    ax.set_title('Edge Distribution')
    ax.legend(fontsize=9)

    ax = axes[1]
    ledger['book_decimal_odds'].hist(bins=30, ax=ax, color='#1f77b4', edgecolor='white')
    ax.set_xlabel('Book Decimal Odds')
    ax.set_title('Odds Distribution at Bet Time')

    ax = axes[2]
    win_mask = ledger['result'] == 'W'
    if win_mask.any():
        ax.scatter(ledger.loc[win_mask, 'edge'], ledger.loc[win_mask, 'pnl'],
                   alpha=0.5, color='#2ca02c', label='Win', s=20)
    loss_mask = ledger['result'] == 'L'
    if loss_mask.any():
        ax.scatter(ledger.loc[loss_mask, 'edge'], ledger.loc[loss_mask, 'pnl'],
                   alpha=0.5, color='#d62728', label='Loss', s=20)
    ax.axhline(0, color='grey', lw=0.8)
    ax.set_xlabel('Edge')
    ax.set_ylabel('PnL ($)')
    ax.set_title('Edge vs PnL')
    ax.legend(fontsize=9)

    plt.tight_layout()
    plt.show()
else:
    print('No bets to analyse.')

## 3.6 Bet-Timing Analysis

In [ ]:
if len(ledger) > 0:
    fig, ax = plt.subplots(figsize=(10, 4))

    # Bin bets by game clock
    ledger['quarter_approx'] = pd.cut(
        ledger['elapsed_seconds'],
        bins=[0, 900, 1800, 2700, 3600, 9999],
        labels=['Q1', 'Q2', 'Q3', 'Q4', 'OT']
    )
    q_counts = ledger.groupby('quarter_approx', observed=True)['pnl'].agg(['count', 'sum', 'mean'])

    bars = ax.bar(q_counts.index, q_counts['count'], color='#1f77b4', edgecolor='white')
    ax2 = ax.twinx()
    ax2.plot(q_counts.index, q_counts['mean'], color='#d62728', marker='o', lw=2, label='Mean PnL')
    ax2.axhline(0, color='grey', lw=0.5)
    ax.set_xlabel('Quarter')
    ax.set_ylabel('Number of Bets')
    ax2.set_ylabel('Mean PnL ($)', color='#d62728')
    ax.set_title('Bet Frequency and Average PnL by Quarter')
    ax2.legend(fontsize=9)

    plt.tight_layout()
    plt.show()

    print(q_counts)

---

## Summary

The full pipeline demonstrates:

1. **Hawkes process** captures the clustering of odds moves around significant game events.
2. **Kalman filter** successfully reduces noise in the book's implied probability signal.
3. **Monte Carlo simulation** provides distributional fair value estimates that detect systematic mispricings.
4. **Fractional Kelly** with exposure caps produces a conservative, drawdown-aware position-sizing scheme.

To extend this to real performance measurement:
- Source live or historical in-play odds from a commercial data provider (e.g. Sportradar, OddsAPI, Pinnacle API).
- Replace `SyntheticOddsGenerator` with a real odds feed.
- Retrain models on a rolling window (online Kalman filter, periodic Hawkes re-estimation).
- Add transaction costs (vig, timing slippage) to the backtest engine.